# biigletools - Python quickstart

Try each function of the Python package on your own BIIGLE data:

1. download a volume report into a table
2. check whether a label exists in a project
3. **preview** which label assignments a deletion would remove (dry run - nothing is deleted)

Everything here only *reads* from BIIGLE. The delete step runs with `dry_run=True`, so no request is sent.

## 0. Install

Run once in a terminal (or uncomment the line below):

```bash
pip install "git+https://github.com/Npiechaud/biigletools.git#subdirectory=python"
```

In [ ]:
# %pip install "git+https://github.com/Npiechaud/biigletools.git#subdirectory=python"

import biigletools
from biigletools import REPORT_TYPES, delete_biigle_label_list, get_biigle_report, label_in_project

print("biigletools loaded. Report types:", REPORT_TYPES)

## 1. Credentials

The functions read `BIIGLE_LOGIN` (your BIIGLE email) and `BIIGLE_TOKEN` (an API token from
*BIIGLE > Settings > Tokens*). If they are not set, this cell asks for them.
The token is typed hidden and is **not** saved in the notebook.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("BIIGLE_LOGIN"):
    os.environ["BIIGLE_LOGIN"] = input("BIIGLE email: ")
if not os.environ.get("BIIGLE_TOKEN"):
    os.environ["BIIGLE_TOKEN"] = getpass("BIIGLE API token: ")

print("Login:", os.environ["BIIGLE_LOGIN"])
print("Token: set" if os.environ["BIIGLE_TOKEN"] else "Token: MISSING")

## 2. Settings

Change these to a volume and a project you have access to.
The IDs are in the BIIGLE URLs: `biigle.de/volumes/<VOLUME_ID>` and `biigle.de/projects/<PROJECT_ID>`.

In [ ]:
VOLUME_ID = 12345            # a volume you can open in BIIGLE
PROJECT_ID = 678             # the project that volume belongs to
LABEL = "Kolga"              # a label name (case-insensitive) or a numeric label ID
BASE_URL = "https://biigle.de/api/v1"

## 3. Download a volume report

`get_biigle_report()` asks BIIGLE to build the report, waits until it is ready, and returns it as a
pandas DataFrame. Large volumes can take a minute or two.

In [ ]:
report = get_biigle_report(VOLUME_ID, type="csv", base_url=BASE_URL)

print(f"{len(report)} rows, {report.shape[1]} columns")
report.head()

In [ ]:
# Annotations per label, most common first
report["label_name"].value_counts().head(15).rename_axis("label").to_frame("annotations")

To keep the CSV file instead of loading it, use `read=False`. It returns the path of the saved file.

In [ ]:
path = get_biigle_report(VOLUME_ID, type="csv", read=False, out_dir="reports", base_url=BASE_URL, verbose=False)
print("Saved to", path)

## 4. Is a label in the project?

`label_in_project()` looks through every label tree attached to the project.

In [ ]:
res = label_in_project(PROJECT_ID, LABEL, base_url=BASE_URL)

print("Found:", res["found"])
res["matches"]

## 5. Preview a deletion (dry run)

`delete_biigle_label_list()` takes a table with an `annotation_label_id` (or `image_label_id`) column.
Here it gets the annotations of `LABEL` from the report above, and **only shows** what would be deleted.

> **Warning:** with `dry_run=False` the label assignments are really deleted in BIIGLE and cannot be
> restored. Without `confirm_required=False` it asks you to type `DELETE` and `YES` first.

In [ ]:
to_delete = report.loc[report["label_name"].str.lower() == str(LABEL).lower(), ["annotation_label_id"]]

preview = delete_biigle_label_list(to_delete, dry_run=True, confirm_required=False, base_url=BASE_URL)
preview.head()